In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import polars as pl
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
import lightgbm as lgb
from metric import score
from scipy.stats import rankdata 
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from lifelines import CoxPHFitter, KaplanMeierFitter, NelsonAalenFitter
from sklearn.preprocessing import StandardScaler

import plotly.io as pio
pio.renderers.default = 'iframe'
pd.options.display.max_columns = None

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... - done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=23c1cf367b831aa49e77424a644e19514fbd5d6d3963d98415c56e4ba6355790
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
# Cell 1: Update CFG class
class CFG:
    train_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/train.csv')
    test_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')
    subm_path = Path('/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv')
    colorscale = 'Sunset'
    color = '#EADDCA'
    batch_size = 32768
    early_stop = 300
    penalizer = 0.01
    n_splits = 5

    # Original CatBoost params
    ctb_params = {
        'iterations': 100,
        'loss_function': 'RMSE',
        'learning_rate': 0.03,
        'random_state': 42,
        'task_type': 'CPU',
        'subsample': 0.85,
        'reg_lambda': 8.0,
        'depth': 8
    }

    # Original LightGBM params
    lgb_params = {
        'objective': 'regression',
        'min_child_samples': 20,
        'num_iterations': 100,
        'learning_rate': 0.01,
        'extra_trees': True,
        'reg_lambda': 3.0,
        'reg_alpha': 0.1,
        'num_leaves': 64,
        'metric': 'rmse',
        'max_depth': 10,
        'device': 'cpu',
        'max_bin': 255,
        'verbose': -1,
        'seed': 42
    }

    # Cox model params
    cox1_params = {
        'grow_policy': 'Depthwise',
        'min_child_samples': 8,
        'loss_function': 'Cox',
        'learning_rate': 0.03,
        'random_state': 42,
        'task_type': 'CPU',
        'num_trees': 6000,
        'subsample': 0.6,  
        'reg_lambda': 8.0,
        'depth': 8,
    }

    cox2_params = {
        'grow_policy': 'Lossguide',
        'loss_function': 'Cox',
        'learning_rate': 0.03,
        'random_state': 42,
        'task_type': 'CPU',
        'num_trees': 6000,
        'subsample': 0.6,  
        'reg_lambda': 8.0,
        'num_leaves': 32,
        'depth': 8,
    }

    cox3_params = {
        'grow_policy': 'Depthwise',
        'min_child_samples': 16,
        'loss_function': 'Cox',
        'learning_rate': 0.02,
        'random_state': 42,
        'task_type': 'CPU',
        'num_trees': 7000,
        'subsample': 0.5,  
        'reg_lambda': 6.0,
        'depth': 10,
    }

In [3]:
class FE:
    def __init__(self, batch_size):
        self.batch_size = batch_size
        self.scaler = StandardScaler()

    def load_data(self, path):
        return pl.read_csv(path, batch_size=self.batch_size)

    def cast_datatypes(self, df):
        num_cols = [
            'hla_high_res_8', 'hla_low_res_8', 'hla_high_res_6',
            'hla_low_res_6', 'hla_high_res_10', 'hla_low_res_10',
            'hla_match_dqb1_high', 'hla_match_dqb1_low',
            'hla_match_drb1_high', 'hla_match_drb1_low',
            'hla_nmdp_6', 'year_hct', 'hla_match_a_high',
            'hla_match_a_low', 'hla_match_b_high', 'hla_match_b_low',
            'hla_match_c_high', 'hla_match_c_low', 'donor_age',
            'age_at_hct', 'comorbidity_score', 'karnofsky_score',
            'efs', 'efs_time'
        ]

        for col in df.columns:
            if col in num_cols:
                df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Float32))  

            else:
                df = df.with_columns(pl.col(col).fill_null('Unknown').cast(pl.String))  

        return df.with_columns(pl.col('ID').cast(pl.Int32))

    def add_features(self, df):
        # Interactions
        df['age_karnofsky'] = df['age_at_hct'] * df['karnofsky_score']
        df['age_comorbidity'] = df['age_at_hct'] * df['comorbidity_score']
        df['donor_recipient_age_diff'] = abs(df['donor_age'] - df['age_at_hct'])
        
        # Time-based features
        df['years_since_2000'] = df['year_hct'] - 2000
        
        # HLA match ratios
        df['hla_match_ratio'] = (df['hla_high_res_8'] + df['hla_low_res_8']) / 16
        
        # Polynomial features for important numerical columns
        df['age_squared'] = df['age_at_hct'] ** 2
        df['karnofsky_squared'] = df['karnofsky_score'] ** 2
        
        return df

    def normalize_features(self, df, train=True):
        num_cols = df.select_dtypes(include=['float32', 'float64', 'int32', 'int64']).columns
        num_cols = [col for col in num_cols if col not in ['ID', 'efs', 'efs_time']]
        
        if train:
            df[num_cols] = self.scaler.fit_transform(df[num_cols])
        else:
            df[num_cols] = self.scaler.transform(df[num_cols])
        
        return df

    def info(self, df):     
        print(f'\nShape of dataframe: {df.shape}')    
        mem = df.memory_usage().sum() / 1024**2
        print('Memory usage: {:.2f} MB\n'.format(mem))
        display(df.head())

    def apply_fe(self, path):
        df = self.load_data(path)
        df = self.cast_datatypes(df)
        df = df.to_pandas()
        df = self.add_features(df)
        df = self.normalize_features(df, train=True)
        self.info(df)
        
        cat_cols = [col for col in df.columns if df[col].dtype == pl.String]
        return df, cat_cols

In [4]:
fe = FE(CFG.batch_size)
train_data, cat_cols = fe.apply_fe(CFG.train_path)
test_data, _ = fe.apply_fe(CFG.test_path)


Shape of dataframe: (28800, 67)
Memory usage: 11.21 MB



,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,age_karnofsky,age_comorbidity,donor_recipient_age_diff,years_since_2000,hla_match_ratio,age_squared,karnofsky_squared
0,0,N/A - non-malignant indication,No,Unknown,No,-2.131316,-1.816076,No TBI,No,0.688781,Bone marrow,No,No,No,IEA,0.771727,+/+,-1.615251,0.671281,Unknown,0.711593,0.557224,No,0.597101,0.597259,BM,Unknown,Unknown,Not Hispanic or Latino,0.260175,No,Unknown,Yes,Unknown,0.664323,No,-2.246948,No,0.588289,No,-1.358153,0.595088,FKalone,No,M-F,0.659817,More than one race,-0.825620,0.48149,No,Unknown,Unrelated,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,42.355999,-1.212271,-0.670055,-0.588860,0.260175,-0.691315,-1.223261,0.547728
1,1,Intermediate,No,Intermediate,No,0.623459,0.785516,"TBI +- Other, >cGy",No,0.688781,Peripheral blood,No,No,No,AML,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,PB,Intermediate,MAC,Not Hispanic or Latino,-2.276400,No,Positive,No,Permissive,0.664323,No,1.791275,No,0.588289,No,0.238416,0.595088,Other GVHD Prophylaxis,No,F-F,0.659817,Asian,0.668652,0.48149,No,Permissive mismatched,Related,"N/A, Mel not given",0.697487,No,0.626518,Yes,0.748055,1.0,4.672000,0.454690,0.521604,0.605512,-2.276400,0.795902,-0.021182,0.547728
2,2,N/A - non-malignant indication,No,Unknown,No,0.623459,0.785516,No TBI,No,0.688781,Bone marrow,No,No,No,HIS,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,BM,Unknown,Unknown,Not Hispanic or Latino,1.211391,No,Unknown,Yes,Unknown,0.664323,No,-2.246948,No,0.588289,No,-0.220651,0.595088,Cyclophosphamide alone,No,F-M,0.659817,More than one race,-0.825620,0.48149,No,Permissive mismatched,Related,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,19.792999,-0.024618,-0.670055,1.039583,1.211391,0.795902,-0.521803,0.547728
3,3,High,No,Intermediate,No,0.623459,0.785516,No TBI,No,0.688781,Bone marrow,No,No,No,ALL,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,BM,Intermediate,MAC,Not Hispanic or Latino,-1.959328,No,Positive,No,Permissive,0.664323,No,-0.581299,No,0.588289,No,0.216664,0.595088,FK+ MMF +- others,No,M-M,0.659817,White,-0.825620,0.48149,Yes,Permissive mismatched,Unrelated,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,102.348999,0.431978,-0.670055,-0.380829,-1.959328,0.795902,-0.047727,0.547728
4,4,High,No,Unknown,No,0.623459,0.785516,No TBI,No,0.688781,Peripheral blood,No,No,No,MPN,0.771727,+/+,0.843860,0.671281,Unknown,0.302018,0.557224,No,0.597101,0.597259,PB,Unknown,MAC,Hispanic or Latino,0.894319,No,Unknown,Yes,Unknown,0.664323,No,0.938338,No,0.588289,No,-0.421955,0.595088,TDEPLETION +- other,No,M-F,0.659817,American Indian or Alaska Native,-0.327529,0.48149,No,Permissive mismatched,Related,MEL,0.697487,No,0.626518,No,0.748055,0.0,16.223000,-0.234796,-0.399759,0.502952,0.894319,0.795902,-0.701873,0.547728



Shape of dataframe: (3, 65)
Memory usage: 0.00 MB



,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,age_karnofsky,age_comorbidity,donor_recipient_age_diff,years_since_2000,hla_match_ratio,age_squared,karnofsky_squared
0,28800,N/A - non-malignant indication,No,Unknown,No,-1.414214,-1.414214,No TBI,No,0.0,Bone marrow,No,No,No,IEA,0.0,+/+,-1.414214,0.0,Unknown,0.0,0.0,No,0.0,0.0,BM,Unknown,Unknown,Not Hispanic or Latino,0.358979,No,Unknown,Yes,Unknown,0.0,No,-0.707107,No,0.0,No,-1.357953,0.0,FKalone,No,M-F,0.0,More than one race,-0.707107,0.0,No,Unknown,Unrelated,"N/A, Mel not given",0.0,No,0.0,No,0.0,-1.357953,-0.707107,-1.366573,0.358979,-1.414214,-1.286980,0.0
1,28801,Intermediate,No,Intermediate,No,0.707107,0.707107,"TBI +- Other, >cGy",No,0.0,Peripheral blood,No,No,No,AML,0.0,+/+,0.707107,0.0,P/P,0.0,0.0,No,0.0,0.0,PB,Intermediate,MAC,Not Hispanic or Latino,-1.364120,No,Positive,No,Permissive,0.0,No,1.414214,No,0.0,No,1.020989,0.0,Other GVHD Prophylaxis,No,F-F,0.0,Asian,1.414214,0.0,No,Permissive mismatched,Related,"N/A, Mel not given",0.0,No,0.0,Yes,0.0,1.020989,1.414214,0.368075,-1.364120,0.707107,1.151193,0.0
2,28802,N/A - non-malignant indication,No,Unknown,No,0.707107,0.707107,No TBI,No,0.0,Bone marrow,No,No,No,HIS,0.0,+/+,0.707107,0.0,P/P,0.0,0.0,No,0.0,0.0,BM,Unknown,Unknown,Not Hispanic or Latino,1.005141,No,Unknown,Yes,Unknown,0.0,No,-0.707107,No,0.0,No,0.336963,0.0,Cyclophosphamide alone,No,F-M,0.0,More than one race,-0.707107,0.0,No,Permissive mismatched,Related,"N/A, Mel not given",0.0,No,0.0,No,0.0,0.336963,-0.707107,0.998498,1.005141,0.707107,0.135787,0.0


In [5]:
train_data

,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,age_karnofsky,age_comorbidity,donor_recipient_age_diff,years_since_2000,hla_match_ratio,age_squared,karnofsky_squared
0,0,N/A - non-malignant indication,No,Unknown,No,-2.131316,-1.816076,No TBI,No,0.688781,Bone marrow,No,No,No,IEA,0.771727,+/+,-1.615251,0.671281,Unknown,0.711593,0.557224,No,0.597101,0.597259,BM,Unknown,Unknown,Not Hispanic or Latino,0.260175,No,Unknown,Yes,Unknown,0.664323,No,-2.246948,No,0.588289,No,-1.358153,0.595088,FKalone,No,M-F,0.659817,More than one race,-0.825620,0.481490,No,Unknown,Unrelated,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,42.355999,-1.212271,-0.670055,-0.588860,0.260175,-0.691315,-1.223261,0.547728
1,1,Intermediate,No,Intermediate,No,0.623459,0.785516,"TBI +- Other, >cGy",No,0.688781,Peripheral blood,No,No,No,AML,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,PB,Intermediate,MAC,Not Hispanic or Latino,-2.276400,No,Positive,No,Permissive,0.664323,No,1.791275,No,0.588289,No,0.238416,0.595088,Other GVHD Prophylaxis,No,F-F,0.659817,Asian,0.668652,0.481490,No,Permissive mismatched,Related,"N/A, Mel not given",0.697487,No,0.626518,Yes,0.748055,1.0,4.672000,0.454690,0.521604,0.605512,-2.276400,0.795902,-0.021182,0.547728
2,2,N/A - non-malignant indication,No,Unknown,No,0.623459,0.785516,No TBI,No,0.688781,Bone marrow,No,No,No,HIS,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,BM,Unknown,Unknown,Not Hispanic or Latino,1.211391,No,Unknown,Yes,Unknown,0.664323,No,-2.246948,No,0.588289,No,-0.220651,0.595088,Cyclophosphamide alone,No,F-M,0.659817,More than one race,-0.825620,0.481490,No,Permissive mismatched,Related,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,19.792999,-0.024618,-0.670055,1.039583,1.211391,0.795902,-0.521803,0.547728
3,3,High,No,Intermediate,No,0.623459,0.785516,No TBI,No,0.688781,Bone marrow,No,No,No,ALL,0.771727,+/+,0.843860,0.671281,P/P,0.711593,0.557224,No,0.597101,0.597259,BM,Intermediate,MAC,Not Hispanic or Latino,-1.959328,No,Positive,No,Permissive,0.664323,No,-0.581299,No,0.588289,No,0.216664,0.595088,FK+ MMF +- others,No,M-M,0.659817,White,-0.825620,0.481490,Yes,Permissive mismatched,Unrelated,"N/A, Mel not given",0.697487,No,0.626518,No,0.748055,0.0,102.348999,0.431978,-0.670055,-0.380829,-1.959328,0.795902,-0.047727,0.547728
4,4,High,No,Unknown,No,0.623459,0.785516,No TBI,No,0.688781,Peripheral blood,No,No,No,MPN,0.771727,+/+,0.843860,0.671281,Unknown,0.302018,0.557224,No,0.597101,0.597259,PB,Unknown,MAC,Hispanic or Latino,0.894319,No,Unknown,Yes,Unknown,0.664323,No,0.938338,No,0.588289,No,-0.421955,0.595088,TDEPLETION +- other,No,M-F,0.659817,American Indian or Alaska Native,-0.327529,0.481490,No,Permissive mismatched,Related,MEL,0.697487,No,0.626518,No,0.748055,0.0,16.223000,-0.234796,-0.399759,0.502952,0.894319,0.795902,-0.701873,0.547728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28795,28795,Intermediate - TED AML case <missing cytogenetics,Unknown,Favorable,No,0.623459,0.785516,No TBI,No,0.688781

In [6]:
class MD:
    def __init__(self, early_stop, penalizer, n_splits, color):
        self.early_stop = early_stop
        self.penalizer = penalizer
        self.n_splits = n_splits
        self.color = color

    def create_target1(self, data, cat_cols):
        cph_data = pd.get_dummies(data, columns=cat_cols, drop_first=True)
        cph = CoxPHFitter(penalizer=self.penalizer)
        cph.fit(cph_data, duration_col='efs_time', event_col='efs')
        data['target1'] = cph.predict_partial_hazard(cph_data)
        return data

    def create_target2(self, data):
        kmf = KaplanMeierFitter()
        kmf.fit(durations=data['efs_time'], event_observed=data['efs'])
        data['target2'] = kmf.survival_function_at_times(data['efs_time']).values
        return data

    def create_target3(self, data):
        naf = NelsonAalenFitter()
        naf.fit(durations=data['efs_time'], event_observed=data['efs'])
        data['target3'] = naf.cumulative_hazard_at_times(data['efs_time']).values
        data['target3'] = 1-data['target3']
        return data

    def create_target4(self, data):
        data['target4'] = data.efs_time.copy()
        data.loc[data.efs == 0, 'target4'] *= -1
        return data

    def train_model(self, data, cat_cols, params, target, title):
        for col in cat_cols:
            data[col] = data[col].astype('category')
            
        X = data.drop(['ID', 'efs', 'efs_time', 'target1', 'target2', 'target3', 'target4'], axis=1)
        y = data[target]
        
        models, fold_scores = [], []
        
        # Use KFold 
        cv = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        
        oof_preds = np.zeros(len(X))
        
        for fold, (train_index, valid_index) in enumerate(cv.split(X)):
            X_train = X.iloc[train_index]
            X_valid = X.iloc[valid_index]
            y_train = y.iloc[train_index]
            y_valid = y.iloc[valid_index]
            
            if title.startswith('LightGBM'):
                model = lgb.LGBMRegressor(**params)
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_valid, y_valid)],
                    eval_metric='rmse',
                    callbacks=[
                        lgb.early_stopping(self.early_stop, verbose=0),
                        lgb.log_evaluation(0)
                    ]
                )
                
            elif title.startswith('CatBoost'):
                model = CatBoostRegressor(**params, verbose=0, cat_features=cat_cols)
                model.fit(
                    X_train, y_train,
                    eval_set=(X_valid, y_valid),
                    early_stopping_rounds=self.early_stop,
                    verbose=0
                )
                
            models.append(model)
            oof_preds[valid_index] = model.predict(X_valid)
            
            y_true_fold = data.iloc[valid_index][['ID', 'efs', 'efs_time', 'race_group']].copy()
            y_pred_fold = data.iloc[valid_index][['ID']].copy()
            y_pred_fold['prediction'] = oof_preds[valid_index]
            
            fold_score = score(y_true_fold, y_pred_fold, 'ID')
            fold_scores.append(fold_score)
        
        y_true = data[['ID', 'efs', 'efs_time', 'race_group']].copy()
        y_pred = data[['ID']].copy()
        y_pred['prediction'] = oof_preds
        
        c_index_score = score(y_true.copy(), y_pred.copy(), 'ID')
        if target == 'target1':
            t = 'Cox Target'
        elif target == 'target2':
            t = 'Kaplan-Meier Target'
        elif target == 'target3':
            t = 'Nelson-Aalen Target'
        else:
            t = 'Cox Loss'
        print(f'\nOverall C-Index for {title} {t}: {c_index_score:.3f}\n')
        
        return models, oof_preds

    def infer_model(self, data, cat_cols, models):
        data = data.drop(['ID'], axis=1)
        for col in cat_cols:
            data[col] = data[col].astype('category')
        return np.mean([model.predict(data) for model in models], axis=0)

In [7]:
md = MD(CFG.early_stop, CFG.penalizer, CFG.n_splits, CFG.color)

# Create all targets
train_data = md.create_target1(train_data, cat_cols)
train_data = md.create_target2(train_data)
train_data = md.create_target3(train_data)
train_data = md.create_target4(train_data)

In [8]:
# Train CatBoost models
print("Training CatBoost models...")
ctb1_models, _ = md.train_model(train_data, cat_cols, CFG.ctb_params, target='target1', title='CatBoost1')

# Train LightGBM models
print("\nTraining LightGBM models...")
lgb1_models, _ = md.train_model(train_data, cat_cols, CFG.lgb_params, target='target1', title='LightGBM1')

# CatBoost predictions
ctb1_preds = md.infer_model(test_data, cat_cols, ctb1_models)
# # LightGBM predictions
lgb1_preds = md.infer_model(test_data, cat_cols, lgb1_models)


Training CatBoost models...

Overall C-Index for CatBoost1 Cox Target: 0.648


Training LightGBM models...

Overall C-Index for LightGBM1 Cox Target: 0.645



In [9]:
# Combine all predictions
preds = [
    ctb1_preds,
    lgb1_preds
]

# Define weights based on model performance
weights = [
    0.55,  # CatBoost weights
    0.55   # Cox weights 
]

# Create ranked predictions
ranked_preds = np.array([rankdata(p) for p in preds])
ensemble_preds = np.sum([w * p for w, p in zip(weights, ranked_preds)], axis=0)

In [10]:
# Create submission
subm_data = pd.read_csv(CFG.subm_path)
subm_data['prediction'] = ensemble_preds
subm_data.to_csv('submission.csv', index=False)
display(subm_data.head())

,ID,prediction
0,28800,1.375
1,28801,3.300
2,28802,1.925
